# Consigna del desafío 4

Replicar y extender el traductor:
- Replicar el modelo en PyTorch.
- Extender el entrenamiento a más datos y tamaños de 
secuencias mayores.
- Explorar el impacto de la cantidad de neuronas en 
las capas recurrentes.
- Mostrar 5 ejemplos de traducciones generadas.
- Extras que se pueden probar: Embeddings 
pre-entrenados para los dos idiomas; cambiar la 
estrategia de generación (por ejemplo muestreo 
aleatorio); 

---
# Imports
---

In [ ]:
import numpy as np
import pandas as pd
import re

import torch
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

c:\Users\julia\anaconda3\envs\PLN\Lib\site-packages\torchtext\__init__.py:7: SyntaxWarning: invalid escape sequence '\ '
  "\n/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ \n"


OSError: [WinError 127] No se encontró el proceso especificado

---
# Dataset
--- 

In [ ]:
text_file = "desafios/spa-eng/spa.txt"

with open(text_file) as f:
    lines = f.read().split("\n")[:-1]

MAX_NUM_SENTENCES = 6000

np.random.seed([40])
np.random.shuffle(lines)

input_sentences = []
output_sentences = []
output_sentences_inputs = []
count = 0

for line in lines:
    count += 1
    if count > MAX_NUM_SENTENCES:
        break

    if '\t' not in line:
        continue

    input_sentence, output = line.rstrip().split('\t')

    output_sentence = output + ' <eos>'
    output_sentences_input = '<sos> ' + output

    input_sentences.append(input_sentence)
    output_sentences.append(output_sentence)
    output_sentences_inputs.append(output_sentences_input)

print("Cantidad de rows disponibles:", len(lines))
print("Cantidad de rows utilizadas:", len(input_sentences))

In [ ]:
input_sentences[0], output_sentences[0], output_sentences_inputs[0]

---
# Preprocesamiento
---

## Tokenizador ingles

In [ ]:
MAX_VOCAB_SIZE = 8000

In [ ]:
input_tokenizer = get_tokenizer('basic_english')

In [ ]:
def yield_tokens(data_iter, tokenizer):
    for text in data_iter:
        yield tokenizer(text)

In [ ]:
vocab = build_vocab_from_iterator(
    yield_tokens(input_sentences),
    max_tokens=MAX_VOCAB_SIZE,         
    specials=['<unk>']                 
)

In [ ]:
vocab.set_default_index(vocab['<unk>'])

In [ ]:
input_integer_seq = [vocab(input_tokenizer(text)) for text in input_sentences]

In [ ]:
word2idx_inputs = vocab.get_stoi()
print(f"Palabras en el vocabulario: {len(word2idx_inputs)}")
print("Diccionario (word_index):", word2idx_inputs)


In [ ]:
max_input_len = max(len(sen) for sen in input_integer_seq)
print("\nSentencia de entrada más larga:", max_input_len)

print("\nSecuencias de enteros generadas:")
print(input_integer_seq)

## Tokenizador español

In [ ]:
def custom_filter_tokenizer(text):
    clean_text = re.sub(r'[^a-zA-Z0-9<> ]', '', text.lower())
    
    # Dividir el texto limpio en tokens por espacio
    return clean_text.split()

In [ ]:
vocab = build_vocab_from_iterator(
    yield_tokens(output_sentences),
    max_tokens=MAX_VOCAB_SIZE,
    specials=['<unk>', '<sos>', '<eos>']
)
vocab.set_default_index(vocab['<unk>'])

In [ ]:
output_integer_seq = [vocab(custom_filter_tokenizer(s)) for s in output_sentences]
output_input_integer_seq = [vocab(custom_filter_tokenizer(s)) for s in output_sentences_inputs]

In [ ]:
word2idx_outputs = vocab.get_stoi()

In [ ]:
print("Palabras en el vocabulario:", len(word2idx_outputs))
print("Diccionario (word_index):", word2idx_outputs)

num_words_output = len(vocab)
print("Tamaño final del vocabulario:", num_words_output)

max_out_len = max(len(sen) for sen in output_integer_seq)
print("Sentencia de salida más larga:", max_out_len)

print("\nSecuencias de enteros (output):", output_integer_seq)
print("Secuencias de enteros (input):", output_input_integer_seq)